# Minimum Detectable Effect (MDE) Table

Per-comparison minimum detectable AUC at 80% power, raw α=0.05, for the manuscript's main-text MDE
table, following the Methods description (two-sample normal-shift Monte Carlo, 3,000 iterations per
candidate effect size, 80% target power). Sample sizes are the achieved cohort sizes printed by
`composite_figures.ipynb` (cell 3), so they match the groups used in the results figures and tables.

**Model.** Group A ~ N(d, 1), Group B ~ N(0, 1), independent, with n_A and n_B matching each
comparison's achieved sample size. `d` is the standardized mean difference (Cohen's d, unit
variance). Significance test: two-sided Mann-Whitney U (matching the test used for the manuscript's
actual AUC/CI results in `composite_figures.ipynb` section 3 — `mannwhitneyu(a, b,
alternative='two-sided')` — so the MDE simulation uses the same test as the analysis it's powering).

**AUC conversion.** Under this normal-shift model, for X ~ N(d, sigma^2), Y ~ N(0, sigma^2):
X - Y ~ N(d, 2*sigma^2), so AUC = P(X > Y) = P(X - Y > 0) = Phi(d / (sqrt(2)*sigma)). With sigma=1
(z-scored features), **AUC = Phi(d / sqrt(2))** — this is the manuscript's stated conversion,
confirmed here and used directly (not re-derived via bootstrap) once the MDE `d` is found.

**Search.** For each comparison, sweep a grid of candidate `d` (step 0.005, range 0.01-2.0). At each
`d`, draw 3,000 independent (Group A, Group B) datasets, run the Mann-Whitney test on each, and record
empirical power (fraction with p < 0.05). The MDE is the `d` where the power curve crosses 80%, found
by linear interpolation between the two bracketing grid points.

A second column gives the MDE at α=0.05/18, the Benjamini-Hochberg threshold for a single true effect in the
largest correction family (18 tract features). Per-family Bonferroni MDEs (α=0.05/7 for the 7 morphometry lobar
composites) are not reported — Bonferroni doesn't match how significance is actually called elsewhere
in the manuscript (FDR/Benjamini-Hochberg), and an FDR-consistent MDE isn't a single well-defined
number the way a Bonferroni one is (it depends on the true-vs-null mixture across the rest of each
feature family). Only the raw, uncorrected MDE is reported here.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, norm

N_ITER = 3000
TARGET_POWER = 0.80
ALPHA_RAW = 0.05
# Corrected threshold: one true effect among m tests in the largest FDR family (tractometry, 9 tracts x FA/MD = 18).
# With a single true effect, its BH threshold is alpha/m.
M_LARGEST_FAMILY = 18
ALPHA_CORR = ALPHA_RAW / M_LARGEST_FAMILY
D_GRID = np.arange(0.01, 2.0, 0.005)

# Achieved sample sizes, as printed by composite_figures.ipynb cell 3:
# "Morphometry -- Epilepsy: 273, FS: 32, Focal: 203, Generalized: 37, DR: 97, DS: 176"
# (tractometry group sizes are identical in that notebook, same subjects, both modalities required)
COMPARISONS = [
    ('Epilepsy vs Functional Seizures', 273, 32, 11),
    ('Focal vs Generalized',            203, 37, 22),
    ('Drug-Resistant vs Drug-Responsive', 97, 176, 33),
]

In [2]:
def power_curve(n_a, n_b, d_grid, alpha, n_iter=N_ITER, base_seed=42):
    """Empirical power at alpha, for each candidate d, from 3,000 independent MWU draws."""
    power = np.empty(len(d_grid))
    for i, d in enumerate(d_grid):
        rng = np.random.default_rng(base_seed * 100_000 + i)  # independent draw per candidate d
        ga = rng.normal(d, 1, size=(n_iter, n_a))
        gb = rng.normal(0, 1, size=(n_iter, n_b))
        _, p = mannwhitneyu(ga, gb, axis=1, alternative='two-sided')
        power[i] = (p < alpha).mean()
    return power

def find_mde_d(d_grid, power_arr, target=TARGET_POWER):
    """Smallest d at which the power curve crosses target, via linear interpolation."""
    idx = np.argmax(power_arr >= target)
    if idx == 0:
        return d_grid[0], True  # power already >= target at grid floor -- flag as suspicious
    d_lo, d_hi = d_grid[idx - 1], d_grid[idx]
    p_lo, p_hi = power_arr[idx - 1], power_arr[idx]
    frac = (target - p_lo) / (p_hi - p_lo)
    return d_lo + frac * (d_hi - d_lo), False

def d_to_auc(d):
    """AUC = P(X > Y) under the normal-shift model, X ~ N(d,1), Y ~ N(0,1) => AUC = Phi(d/sqrt(2))."""
    return norm.cdf(d / np.sqrt(2))

In [3]:
rows = []
for name, n_a, n_b, seed in COMPARISONS:
    power = power_curve(n_a, n_b, D_GRID, ALPHA_RAW, base_seed=seed)
    d_mde, floor_flag = find_mde_d(D_GRID, power)
    power_c = power_curve(n_a, n_b, D_GRID, ALPHA_CORR, base_seed=seed)  # same draws, corrected alpha
    d_mde_c, _ = find_mde_d(D_GRID, power_c)
    rows.append({
        'comparison': name, 'n_A': n_a, 'n_B': n_b,
        'd_raw': d_mde, 'MDE_AUC_raw': d_to_auc(d_mde), '_floor_flag': floor_flag,
        'd_corr': d_mde_c, 'MDE_AUC_corrected': d_to_auc(d_mde_c),
    })

mde_full = pd.DataFrame(rows)
mde_full

,comparison,n_A,n_B,d_raw,MDE_AUC_raw,_floor_flag,d_corr,MDE_AUC_corrected
0,Epilepsy vs Functional Seizures,273,32,0.544773,0.649960,False,0.756154,0.703565
1,Focal vs Generalized,203,37,0.517000,0.642659,False,0.709404,0.692034
2,Drug-Resistant vs Drug-Responsive,97,176,0.362045,0.601026,False,0.505000,0.639487


## d -> AUC mapping used (sanity print)

Confirms the conversion formula (`AUC = Phi(d / sqrt(2))`) against the actual `d` values found by the
search, for every row.

In [4]:
for _, r in mde_full.iterrows():
    d, auc = r['d_raw'], r['MDE_AUC_raw']
    assert abs(norm.cdf(d / np.sqrt(2)) - auc) < 1e-12
    print(f"{r['comparison']:38s} d={d:.4f}  Phi(d/sqrt(2))={norm.cdf(d/np.sqrt(2)):.4f}  AUC_mde={auc:.4f}")

Epilepsy vs Functional Seizures        d=0.5448  Phi(d/sqrt(2))=0.6500  AUC_mde=0.6500
Focal vs Generalized                   d=0.5170  Phi(d/sqrt(2))=0.6427  AUC_mde=0.6427
Drug-Resistant vs Drug-Responsive      d=0.3620  Phi(d/sqrt(2))=0.6010  AUC_mde=0.6010


## Sanity checks

1. **Raw MDEs vs. manuscript's stated values** (>=0.65 for epilepsy-vs-FS and focal-vs-generalized;
   >=0.61 for drug-response) — reported below, discrepancies NOT silently corrected.
2. **Monotonicity**: smaller / more imbalanced groups (epilepsy-vs-FS, focal-vs-generalized; tiny
   second group) should show the largest MDEs; drug-response (largest, most balanced group) should
   show the smallest.

In [5]:
print('=== Check 1: raw MDE vs. manuscript claim ===')
claims = {'Epilepsy vs Functional Seizures': 0.65, 'Focal vs Generalized': 0.65,
          'Drug-Resistant vs Drug-Responsive': 0.61}
for _, r in mde_full.iterrows():
    claim = claims[r['comparison']]
    got = r['MDE_AUC_raw']
    status = 'MATCHES' if got >= claim else f'DOES NOT MATCH (short by {claim - got:.4f})'
    print(f"{r['comparison']:38s} claimed >= {claim:.2f}, simulated = {got:.4f}  -> {status}")

print()
print('=== Check 2: monotonicity (smaller/more imbalanced -> larger MDE) ===')
print(mde_full[['comparison', 'n_A', 'n_B', 'MDE_AUC_raw']].to_string(index=False))

=== Check 1: raw MDE vs. manuscript claim ===
Epilepsy vs Functional Seizures        claimed >= 0.65, simulated = 0.6500  -> DOES NOT MATCH (short by 0.0000)
Focal vs Generalized                   claimed >= 0.65, simulated = 0.6427  -> DOES NOT MATCH (short by 0.0073)
Drug-Resistant vs Drug-Responsive      claimed >= 0.61, simulated = 0.6010  -> DOES NOT MATCH (short by 0.0090)

=== Check 2: monotonicity (smaller/more imbalanced -> larger MDE) ===
                       comparison  n_A  n_B  MDE_AUC_raw
  Epilepsy vs Functional Seizures  273   32     0.649960
             Focal vs Generalized  203   37     0.642659
Drug-Resistant vs Drug-Responsive   97  176     0.601026


## Final table: save CSV + LaTeX/Markdown

In [6]:
out = mde_full[['comparison', 'n_A', 'n_B', 'MDE_AUC_raw', 'MDE_AUC_corrected']].copy()
out['MDE_AUC_raw'] = out['MDE_AUC_raw'].round(3)
out['MDE_AUC_corrected'] = out['MDE_AUC_corrected'].round(3)
out.to_csv('../outputs/mde_table.csv', index=False)
out

,comparison,n_A,n_B,MDE_AUC_raw,MDE_AUC_corrected
0,Epilepsy vs Functional Seizures,273,32,0.650,0.704
1,Focal vs Generalized,203,37,0.643,0.692
2,Drug-Resistant vs Drug-Responsive,97,176,0.601,0.639


In [7]:
md_lines = ['| Comparison | n (group 1) | n (group 2) | MDE AUC (raw \u03b1=0.05) | MDE AUC (corrected, \u03b1=0.05/18) |',
            '|---|---|---|---|---|']
for _, r in out.iterrows():
    md_lines.append(f"| {r['comparison']} | {r['n_A']} | {r['n_B']} | {r['MDE_AUC_raw']:.3f} | {r['MDE_AUC_corrected']:.3f} |")
markdown_table = '\n'.join(md_lines)
print(markdown_table)

| Comparison | n (group 1) | n (group 2) | MDE AUC (raw α=0.05) | MDE AUC (corrected, α=0.05/18) |
|---|---|---|---|---|
| Epilepsy vs Functional Seizures | 273 | 32 | 0.650 | 0.704 |
| Focal vs Generalized | 203 | 37 | 0.643 | 0.692 |
| Drug-Resistant vs Drug-Responsive | 97 | 176 | 0.601 | 0.639 |


In [8]:
latex_lines = [
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Minimum detectable AUC (80\% power, raw $\alpha$=0.05) by comparison.}',
    r'\label{tab:mde}',
    r'\begin{tabular}{lccc}',
    r'\toprule',
    r'Comparison & $n$ (group 1) & $n$ (group 2) & MDE AUC (raw $\alpha$=0.05) \\',
    r'\midrule',
]
for _, r in out.iterrows():
    latex_lines.append(f"{r['comparison']} & {r['n_A']} & {r['n_B']} & {r['MDE_AUC_raw']:.3f} \\\\")
latex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
latex_table = '\n'.join(latex_lines)
print(latex_table)

\begin{table}[t]
\centering
\caption{Minimum detectable AUC (80\% power, raw $\alpha$=0.05) by comparison.}
\label{tab:mde}
\begin{tabular}{lccc}
\toprule
Comparison & $n$ (group 1) & $n$ (group 2) & MDE AUC (raw $\alpha$=0.05) \\
\midrule
Epilepsy vs Functional Seizures & 273 & 32 & 0.650 \\
Focal vs Generalized & 203 & 37 & 0.643 \\
Drug-Resistant vs Drug-Responsive & 97 & 176 & 0.601 \\
\bottomrule
\end{tabular}
\end{table}
